In [3]:
# Read the pdf

file_path = "../Data/odisha_judgement_files/display_judgement20pdf.pdf"
# file_path = "/Users/m2sm/Desktop/projects/Agentic-AI/Judgement_Bot/Data/odisha_judgement_files/display_judgement20pdf.pdf"

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path)

documents = loader.load()
documents

invalid pdf header: b'\r\n\xef\xbb\xbf'
incorrect startxref pointer(1)
parsing for Object Streams


[Document(metadata={'producer': 'GPL Ghostscript 10.01.1; modified using eSign™ 5.5.13.2 ©2000-2020', 'creator': 'PDF24 Creator', 'creationdate': '2023-07-06T11:02:00+05:30', 'moddate': '2023-07-06T11:06:52+05:30', 'source': '../Data/odisha_judgement_files/display_judgement20pdf.pdf', 'total_pages': 25, 'page': 0, 'page_label': '1'}, page_content='CRLMC No.1462 of 2023                                                              Page 1 of 25 \n \n  \n \n           IN THE HIGH COURT OF ORISSA AT CUTTACK \n \nCRLMC NO.1462 OF 2023 \n \n(Application under Section 482 of the Code of Criminal  \nProcedure for fresh investigation or re-investigation by any \nindependent agency) \n \n            \n            Bandhna Toppo    \n                                                   …     Petitioner \n              \n     -versus-  \n \nState of Orissa  \nand others                      …     Opposite Parties \n \n                                                                                    

# Traditional chunking and retrieval



In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=[
            "\n\n",           # Split by major paragraphs first
            "\n",             # Then split by line breaks
            ". ",             # Then split by sentences
            ", ",             # Then split by clauses
            " "               # Last resort: split by words
        ]
)

In [6]:

raw_chunks = text_splitter.split_documents(documents)

In [7]:
len(raw_chunks)

48

In [8]:
type(raw_chunks[0])

langchain_core.documents.base.Document

In [9]:
raw_chunks[0].metadata


{'producer': 'GPL Ghostscript 10.01.1; modified using eSign™ 5.5.13.2 ©2000-2020',
 'creator': 'PDF24 Creator',
 'creationdate': '2023-07-06T11:02:00+05:30',
 'moddate': '2023-07-06T11:06:52+05:30',
 'source': '../Data/odisha_judgement_files/display_judgement20pdf.pdf',
 'total_pages': 25,
 'page': 0,
 'page_label': '1'}

In [10]:
raw_chunks[0].metadata['source'].split('/')[-1].split('.')[0]

'display_judgement20pdf'

In [11]:
# raw_chunks

from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("BAAI/bge-large-en-v1.5")






Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7170.88it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
import chromadb
collection_name: str = "J_files_recursive"
batch_size: int = 250
host: str = "manishs-mac-mini.tailc96719.ts.net"
port: int = 8030

client = chromadb.HttpClient(host=host, port=port)

collection = client.get_or_create_collection(name=collection_name)

chunk_no =0
for chunk in raw_chunks:
    chunk_no += 1
    id = chunk.metadata['source'].split('/')[-1].replace('.pdf','') + "_chunk_" + str(chunk_no)
    source = chunk.metadata['source']
    total_pages = chunk.metadata['total_pages']
    page_number = chunk.metadata['page']
    page_label = chunk.metadata['page_label']
    embedding = encoder.encode(chunk.page_content)
    collection.add(
        ids=id,
        embeddings=embedding,
        documents=chunk.page_content,
        metadatas={'source':source, 'total_pages':total_pages, 'page_number':page_number, 'page_label':page_label}
    )



In [14]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
openai_client = OpenAI()
from sentence_transformers import SentenceTransformer

import chromadb
collection_name: str = "J_files_recursive"
batch_size: int = 250
host: str = "manishs-mac-mini.tailc96719.ts.net"
port: int = 8030

client = chromadb.HttpClient(host=host, port=port)

collection = client.get_or_create_collection(name=collection_name)
encoder = SentenceTransformer("BAAI/bge-large-en-v1.5")


def generate_query_variations(input_query:str):
    prompt = f"""You are an AI legal assistant. Your task is to generate 3 different versions 
    of the given user query to retrieve relevant documents from a vector database. 
    Provide these alternative questions separated by newlines, with no numbering or extra text.
    
    Original Query: {input_query}"""

    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role':'user', 'content':prompt}],
        temperature=0.2,
    )

    variations = response.choices[0].message.content.strip().split('\n')
    return [input_query] + [variation.strip() for variation in variations if variation.strip()] 




Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6397.05it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
from typing import List, Dict
def retrieve_contexts(queries: List[str], top_k: int = 3) -> List[Dict]:
    """
    Embeds all query variations, searches ChromaDB, and deduplicates the results.
    """
    # CRITICAL: BGE models require this prefix for the search queries!
    bge_prefix = "Represent this sentence for searching relevant passages: "
    formatted_queries = [bge_prefix + q for q in queries]
    
    # Generate embeddings for all queries at once
    query_embeddings = encoder.encode(formatted_queries).tolist()
    
    # Search ChromaDB
    results = collection.query(
        query_embeddings=query_embeddings,
        n_results=top_k
    )
    
    # Deduplicate results (since multiple query variations might find the same chunk)
    unique_chunks = {}
    
    # Chroma returns lists of lists (one list per query). We flatten them here.
    for i in range(len(queries)):
        for j in range(len(results['ids'][i])):
            chunk_id = results['ids'][i][j]
            if chunk_id not in unique_chunks:
                unique_chunks[chunk_id] = {
                    "id": chunk_id,
                    "text": results['documents'][i][j],
                    "metadata": results['metadatas'][i][j],
                    "distance": results['distances'][i][j]
                }
                
    # Sort by distance (lower distance = higher similarity in ChromaDB defaults)
    sorted_chunks = sorted(unique_chunks.values(), key=lambda x: x['distance'])
    
    # Return the top N unique chunks overall
    return sorted_chunks[:top_k * 2] 

def generate_final_answer(original_query: str, retrieved_chunks: List[Dict]) -> str:
    """
    Feeds the retrieved chunks to the LLM and forces citation formatting.
    """
    # 1. Format the context block nicely for the LLM
    context_text = ""
    for i, chunk in enumerate(retrieved_chunks):
        meta = chunk['metadata']
        # We give the LLM the exact metadata variables it needs to make citations
        context_text += f"--- Document {i+1} ---\n"
        context_text += f"Case No: {meta.get('case_number', 'Unknown')}\n"
        context_text += f"File Source: {meta.get('source', 'Unknown')}\n"
        context_text += f"Content: {chunk['text']}\n\n"

    # 2. The Strict System Prompt
    system_prompt = """You are an expert Indian Legal AI Assistant. 
    Answer the user's question using ONLY the provided context documents.
    
    RULES:
    1. If the answer is not contained in the context, say "I cannot find the answer in the provided documents." Do not guess.
    2. You MUST cite your sources immediately after stating a fact. 
    3. Use this strict citation format: [Case No: <case_number>, File: <source>]
    
    Example: The petition was dismissed because the principles of natural justice were not violated [Case No: W.P.(C) 1234, File: judgment.pdf]."""

    # 3. Generate Answer
    response = openai_client.chat.completions.create(
        model="gpt-4o", # Use a smarter model for the final reasoning phase
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context Documents:\n{context_text}\n\nUser Question: {original_query}"}
        ],
        temperature=0.1 # Keep it low to prevent hallucinations
    )
    
    return response.choices[0].message.content



In [18]:
#  The user asks a question
# user_query = "On what grounds did the petitioner seek a fresh investigation?"
user_query = "can you summerise the   Bandhna Toppo  case??"
print(f"\n🗣️ USER QUERY: {user_query}")

# Step 1: Multi-Query Generation
print("\n🧠 1. Generating alternative search queries...")
search_queries = generate_query_variations(user_query)
for q in search_queries:
    print(f"   - {q}")
    
# Step 2: Retrieval
print("\n📚 2. Retrieving relevant documents from ChromaDB...")
retrieved_data = retrieve_contexts(search_queries, top_k=2)
print(f"   Found {len(retrieved_data)} unique relevant chunks.")

# Step 3: Generation
print("\n⚖️ 3. Generating legal answer with citations...\n")
final_answer = generate_final_answer(user_query, retrieved_data)

print("================ FINAL ANSWER ================")
print(final_answer)
print("==============================================")


🗣️ USER QUERY: can you summerise the   Bandhna Toppo  case??

🧠 1. Generating alternative search queries...
   - can you summerise the   Bandhna Toppo  case??
   - can you provide a summary of the Bandhna Toppo case?
   - what are the key details of the Bandhna Toppo case?
   - could you give me an overview of the Bandhna Toppo case?

📚 2. Retrieving relevant documents from ChromaDB...
   Found 3 unique relevant chunks.

⚖️ 3. Generating legal answer with citations...

================ FINAL ANSWER ================
The case involves a petition filed by Bandhna Toppo under Section 482 of the Code of Criminal Procedure, seeking a fresh investigation or re-investigation by an independent agency. The case pertains to the death of Anand Toppo, the petitioner's son, who was found unconscious and later declared dead at Capital Hospital, Bhubaneswar. The petitioner suspected foul play and attempted to lodge a complaint, but it was not accepted as an FIR had already been lodged under Infocity 

In [ ]:
# smart chunking and retrieval




# Advance chunking

